In [1]:
from pyspark.sql import SparkSession

import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("Iceberg-V2-Test")
     .master(master_url)
    .config("hive.metastore.uris", "thrift://localhost:9083")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    .config(
        "spark.sql.catalog.iceberg",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.iceberg.type",
        "hive"
    )
    .config(
        "spark.sql.catalog.iceberg.uri",
        "thrift://localhost:9083"
    )
    .config("spark.sql.warehouse.dir", "hdfs://localhost:9000/user/hive/warehouse") \

    .enableHiveSupport()
    .getOrCreate()
)


sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Catalog       :", spark.conf.get("spark.sql.catalogImplementation"))
print("Warehouse     :", spark.conf.get("spark.sql.warehouse.dir"))
print("Spark UI      :", sc.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/01 11:41:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.9
Spark master  : spark://f8b6936f061c.mylabserver.com:7077
Catalog       : hive
Warehouse     : hdfs://localhost:9000/user/hive/warehouse
Spark UI      : http://f8b6936f061c.mylabserver.com:4040


In [2]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|spark_catalog|
+-------------+



In [3]:
# ============================================================
# 2. Show existing Iceberg namespaces
# ============================================================

spark.sql("SHOW NAMESPACES IN iceberg").show(truncate=False)

+--------------+
|namespace     |
+--------------+
|default       |
|hive_basics   |
|spark_training|
|training      |
+--------------+



In [4]:
# ============================================================
# 3. Create namespace/database
# ============================================================

spark.sql("""
CREATE NAMESPACE IF NOT EXISTS iceberg.demo
""")

DataFrame[]

In [5]:
spark.sql("SHOW NAMESPACES IN iceberg").show(truncate=False)

+--------------+
|namespace     |
+--------------+
|default       |
|demo          |
|hive_basics   |
|spark_training|
|training      |
+--------------+



In [6]:
spark.sql("""
CREATE TABLE iceberg.demo.customers (
    customer_id BIGINT,
    name STRING,
    city STRING,
    amount DECIMAL(10,2)
)
USING iceberg
TBLPROPERTIES (
    'format-version' = '2'
);
""")

DataFrame[]

In [7]:
spark.sql("SHOW TABLEs IN iceberg.demo").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|demo     |customers|false      |
+---------+---------+-----------+



In [8]:

print("=== Description ===")
spark.sql("""
DESCRIBE TABLE EXTENDED iceberg.demo.customers
""").show(100, truncate=False)


=== Description ===
+-----------------------------+-------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                              |comment|
+-----------------------------+-------------------------------------------------------------------------------------------------------+-------+
|customer_id                  |bigint                                                                                                 |NULL   |
|name                         |string                                                                                                 |NULL   |
|city                         |string                                                                                                 |NULL   |
|amount                       |decimal(10,2)                                                                        

In [9]:

print("=== Properties ===")
spark.sql("""
SHOW TBLPROPERTIES iceberg.demo.customers
""").show(100, truncate=False)

=== Properties ===
+-------------------------------+---------------+
|key                            |value          |
+-------------------------------+---------------+
|current-snapshot-id            |none           |
|format                         |iceberg/parquet|
|format-version                 |2              |
|write.parquet.compression-codec|zstd           |
+-------------------------------+---------------+



In [10]:
 

spark.sql("""
INSERT INTO iceberg.demo.customers VALUES
    (1, 'Alice', 'Bangalore', CAST(1200.00 AS DECIMAL(10,2))),
    (2, 'Bob', 'Hyderabad',  CAST(1800.00 AS DECIMAL(10,2))),
    (3, 'Carol', 'Chennai',  CAST(950.00 AS DECIMAL(10,2)))
""")

DataFrame[]

In [11]:
spark.sql("""
SELECT *
FROM iceberg.demo.customers
ORDER BY customer_id
""").show(truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

+-----------+-----+---------+-------+
|customer_id|name |city     |amount |
+-----------+-----+---------+-------+
|1          |Alice|Bangalore|1200.00|
|2          |Bob  |Hyderabad|1800.00|
|3          |Carol|Chennai  |950.00 |
+-----------+-----+---------+-------+



In [12]:
spark.sql("""
UPDATE iceberg.demo.customers
SET amount = CAST(1500.00 AS DECIMAL(10,2))
WHERE customer_id = 1
""")

DataFrame[]

In [13]:
spark.sql("""
SELECT *
FROM iceberg.demo.customers
ORDER BY customer_id
""").show(truncate=False)

+-----------+-----+---------+-------+
|customer_id|name |city     |amount |
+-----------+-----+---------+-------+
|1          |Alice|Bangalore|1500.00|
|2          |Bob  |Hyderabad|1800.00|
|3          |Carol|Chennai  |950.00 |
+-----------+-----+---------+-------+



In [14]:

spark.sql("""
DELETE FROM iceberg.demo.customers
WHERE customer_id = 3
""")

DataFrame[]

In [15]:
spark.sql("""
SELECT *
FROM iceberg.demo.customers
ORDER BY customer_id
""").show(truncate=False)

+-----------+-----+---------+-------+
|customer_id|name |city     |amount |
+-----------+-----+---------+-------+
|1          |Alice|Bangalore|1500.00|
|2          |Bob  |Hyderabad|1800.00|
+-----------+-----+---------+-------+



In [16]:
# MERGE INTO
source_df = spark.createDataFrame(
    [
        (1, "Alice Updated", "Bangalore", 2200.00),
        (4, "David", "Pune", 1750.00),
    ],
    ["customer_id", "name", "city", "amount"]
)

source_df.createOrReplaceTempView("customer_updates")

In [17]:
# ============================================================
# 14. MERGE INTO
# ============================================================

spark.sql("""
MERGE INTO iceberg.demo.customers AS target

USING (
    SELECT
        customer_id,
        name,
        city,
        CAST(amount AS DECIMAL(10,2)) AS amount
    FROM customer_updates
) AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.name = source.name,
        target.city = source.city,
        target.amount = source.amount

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        name,
        city,
        amount
    )
    VALUES (
        source.customer_id,
        source.name,
        source.city,
        source.amount
    )
""")

DataFrame[]

In [18]:
spark.sql("""
SELECT *
FROM iceberg.demo.customers
ORDER BY customer_id
""").show(truncate=False)

+-----------+-------------+---------+-------+
|customer_id|name         |city     |amount |
+-----------+-------------+---------+-------+
|1          |Alice Updated|Bangalore|2200.00|
|2          |Bob          |Hyderabad|1800.00|
|4          |David        |Pune     |1750.00|
+-----------+-------------+---------+-------+



In [19]:
# Snapshot

# ============================================================
# 16. Snapshots
# ============================================================

spark.sql("""
SELECT *
FROM iceberg.demo.customers.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                            |summary                        

In [ ]:
spark.sql("""
SELECT *
FROM iceberg.demo.customers.history
ORDER BY made_current_at
""").show(truncate=False)

In [5]:
spark.stop()